*  DSC540-T303 Data Preparation (2265-1)
*  Milestone 2
*  David Jorgensen

# Milestone 2

# Instructions

Perform at least 5 data transformation and/or cleansing steps to your flat file data. The below examples are not required - they are just potential transformations you could do. If your data doesn't work for these scenarios, complete different transformations. You can do the same transformation multiple times if needed to clean your data. The goal is a clean dataset at the end of the milestone.

Examples:

- Replace Headers
- Format data into a more readable format
- Identify outliers and bad data
- Find duplicates
- Fix casing or inconsistent values
- Conduct Fuzzy Matching

Make sure you clearly number and label each transformation step (Step #1, Step #2, etc.) in your code and describe what it is doing in 1-2 sentences.

You must submit the following:

- Jupyter Notebook File or PDF of your code with Milestone # listed.
- Each transformation should be labeled with description or what it is doing.
- Human readable dataset after all transformations should be printed at the end of your notebook.
- 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions (this can be done in your notebook):
    - What changes were made to the data?
    - Are there any legal or regulatory guidelines for your data or project topic?
    - What risks could be created based on the transformations done?
    - Did you make any assumptions in cleaning/transforming the data?
    - How was your data sourced/verified for credibility?
    - Was your data acquired in an ethical way?
    - How would you mitigate any of the ethical implications you have identified?

# Assignment

In [82]:
# Load Libraries
import pandas as pd
import re

In [83]:
# Load data
flat_df = pd.read_csv('./broadway.csv')
# Display the first 10 rows of the dataset
flat_df.head(10)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances
0,26,8/26/1990,8,1990,Tru,Booth,Play,5500,88,134456,0,8
1,24,3/24/1991,3,1991,Miss Saigon,Broadway,Musical,1737,100,100647,0,0
2,31,3/31/1991,3,1991,Miss Saigon,Broadway,Musical,12160,100,634424,0,0
3,7,4/7/1991,4,1991,Miss Saigon,Broadway,Musical,13921,100,713353,0,0
4,14,4/14/1991,4,1991,Miss Saigon,Broadway,Musical,10973,90,573981,0,4
5,21,4/21/1991,4,1991,Miss Saigon,Broadway,Musical,14076,101,706793,0,8
6,28,4/28/1991,4,1991,Miss Saigon,Broadway,Musical,14065,101,714968,0,8
7,5,5/5/1991,5,1991,Miss Saigon,Broadway,Musical,14064,101,730765,0,8
8,12,5/12/1991,5,1991,Miss Saigon,Broadway,Musical,13896,100,766713,0,8
9,19,5/19/1991,5,1991,Miss Saigon,Broadway,Musical,13738,99,763332,0,8


In [84]:
# Step 1: Filter to keep only rows where Show.Type is "Musical"
flat_df = flat_df[flat_df['Show.Type'] == 'Musical']
flat_df.head(10)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances
1,24,3/24/1991,3,1991,Miss Saigon,Broadway,Musical,1737,100,100647,0,0
2,31,3/31/1991,3,1991,Miss Saigon,Broadway,Musical,12160,100,634424,0,0
3,7,4/7/1991,4,1991,Miss Saigon,Broadway,Musical,13921,100,713353,0,0
4,14,4/14/1991,4,1991,Miss Saigon,Broadway,Musical,10973,90,573981,0,4
5,21,4/21/1991,4,1991,Miss Saigon,Broadway,Musical,14076,101,706793,0,8
6,28,4/28/1991,4,1991,Miss Saigon,Broadway,Musical,14065,101,714968,0,8
7,5,5/5/1991,5,1991,Miss Saigon,Broadway,Musical,14064,101,730765,0,8
8,12,5/12/1991,5,1991,Miss Saigon,Broadway,Musical,13896,100,766713,0,8
9,19,5/19/1991,5,1991,Miss Saigon,Broadway,Musical,13738,99,763332,0,8
10,26,5/26/1991,5,1991,Miss Saigon,Broadway,Musical,13897,100,769137,0,8


In [85]:
# Step 2: Standardize String Columns
# Strip whitespace from text columns to ensure consistent grouping
text_cols = flat_df.select_dtypes(include='object').columns
for col in text_cols:
    flat_df[col] = flat_df[col].str.strip()

In [86]:
# Step 3: Ensure Numeric Columns are Correctly Typed
# This prevents errors if attendance/revenue were imported as objects due to commas or symbols
numeric_cols = ['Statistics.Attendance', 'Statistics.Revenue', 'Statistics.Capacity']
for col in numeric_cols:
    if col in flat_df.columns:
        flat_df[col] = pd.to_numeric(flat_df[col], errors='coerce')

In [87]:
# Step 4: Handle Missing Values
# Fill missing numerical values with median to preserve distribution
for col in numeric_cols:
    if col in flat_df.columns:
        median_val = flat_df[col].median()
        # Assign the result of fillna back to the DataFrame column
        flat_df[col] = flat_df[col].fillna(median_val)

In [88]:
# Step 5: Identify Outliers (Keep them, but flag them)
# This preserves the data points while marking them for potential later analysis
Q1 = flat_df['Statistics.Attendance'].quantile(0.25)
Q3 = flat_df['Statistics.Attendance'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

flat_df['IsAttendanceOutlier'] = (flat_df['Statistics.Attendance'] > upper_bound) | \
                                 (flat_df['Statistics.Attendance'] < lower_bound)


In [89]:
# Step 6: Reset Index in case I plan to remove rows later
flat_df.reset_index(drop=True, inplace=True)

In [90]:
# SPOT CHECK: Verify the wrangling
print("Data Types:")
print(flat_df.dtypes)
print("\nMissing Values:")
print(flat_df.isnull().sum())
print("\nOutlier Count:")
print(flat_df['IsAttendanceOutlier'].sum())

Data Types:
Date.Day                       int64
Date.Full                     object
Date.Month                     int64
Date.Year                      int64
Show.Name                     object
Show.Theatre                  object
Show.Type                     object
Statistics.Attendance          int64
Statistics.Capacity            int64
Statistics.Gross               int64
Statistics.Gross Potential     int64
Statistics.Performances        int64
IsAttendanceOutlier             bool
dtype: object

Missing Values:
Date.Day                      0
Date.Full                     0
Date.Month                    0
Date.Year                     0
Show.Name                     0
Show.Theatre                  0
Show.Type                     0
Statistics.Attendance         0
Statistics.Capacity           0
Statistics.Gross              0
Statistics.Gross Potential    0
Statistics.Performances       0
IsAttendanceOutlier           0
dtype: int64

Outlier Count:
12


In [91]:
# Print list of each unique Show.Name, organized in alphabetical order
print("\nUnique Show Names:")
print(sorted(flat_df['Show.Name'].unique()))
print("Number of Unique Shows:", len(flat_df['Show.Name'].unique()))


Unique Show Names:
['110 In The Shade', '13', '1776', "42Nd Street '01", '9 To 5', 'A Catered Affair', "A Chorus Line '06", 'A Christmas Story', 'A Class Act', 'A Funny Thing Happened On The Way To The Forum', "A Gentleman'S Guide To Love And Murder", 'A Little Night Music', 'A Night With Janis Joplin', 'A Tale Of Two Cities', 'A Year With Frog And Toad', 'After Midnight', 'Aida', 'Aladdin', 'All About Me', 'All Shook Up', 'Allegiance', 'Amazing Grace', 'American Idiot', 'American Psycho', 'Amour', 'An American In Paris', 'An Evening With Jerry Herman', 'Annie', "Annie '12", 'Annie Get Your Gun', "Anything Goes '11", 'Assassins', 'Avenue Q', "Baby It'S You!", 'Band In Berlin', 'Beautiful', 'Beauty And The Beast', 'Bells Are Ringing', 'Big', 'Big Fish', "Big River '03", 'Billy Elliot: The Musical', 'Blast!', 'Bloody Bloody Andrew Jackson', 'Bombay Dreams', 'Bonnie And Clyde', 'Bright Star', 'Bring In Da Noise, Bring In Da Funk', 'Bring It On The Musical', 'Brooklyn', 'Bullets Over Broa

In [92]:
# Step 7: Remove date suffix from all shows (" yy", " 'yy", " yyyy")
flat_df['Show.Name'] = flat_df['Show.Name'].str.replace(r'\s\d{2}$|\s\'\d{2}$|\s\d{4}$', '', regex=True)
print("Number of Unique Shows:", len(flat_df['Show.Name'].unique()))

# Step 8: Fix Show Name casing to be title case
def fix_title_casing(text):
    # Apply title casing first
    text = text.title()
    # Lowercase letter after apostrophe (e.g., O'Connors -> O'connors)
    text = re.sub(r"'([A-Z])", lambda m: "'" + m.group(1).lower(), text)
    # Lowercase letter after digit (e.g., 42Nd -> 42nd)
    text = re.sub(r'(\d)([A-Z])', lambda m: m.group(1) + m.group(2).lower(), text)
    return text

flat_df['Show.Name'] = flat_df['Show.Name'].apply(fix_title_casing)

print(sorted(flat_df['Show.Name'].unique()))

Number of Unique Shows: 281
['110 In The Shade', '13', '1776', '42nd Street', '9 To 5', 'A Catered Affair', 'A Chorus Line', 'A Christmas Story', 'A Class Act', 'A Funny Thing Happened On The Way To The Forum', "A Gentleman's Guide To Love And Murder", 'A Little Night Music', 'A Night With Janis Joplin', 'A Tale Of Two Cities', 'A Year With Frog And Toad', 'After Midnight', 'Aida', 'Aladdin', 'All About Me', 'All Shook Up', 'Allegiance', 'Amazing Grace', 'American Idiot', 'American Psycho', 'Amour', 'An American In Paris', 'An Evening With Jerry Herman', 'Annie', 'Annie Get Your Gun', 'Anything Goes', 'Assassins', 'Avenue Q', "Baby It's You!", 'Band In Berlin', 'Beautiful', 'Beauty And The Beast', 'Bells Are Ringing', 'Big', 'Big Fish', 'Big River', 'Billy Elliot: The Musical', 'Blast!', 'Bloody Bloody Andrew Jackson', 'Bombay Dreams', 'Bonnie And Clyde', 'Bright Star', 'Bring In Da Noise, Bring In Da Funk', 'Bring It On The Musical', 'Brooklyn', 'Bullets Over Broadway', 'Burn The Floo

In [93]:
# Print 30 random rows of data
flat_df.sample(n=30, random_state=42)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances,IsAttendanceOutlier
12959,24,2/24/2008,2,2008,Mamma Mia!,Winter Garden,Musical,10516,88,759707,64,8,False
14184,22,3/22/2009,3,2009,Avenue Q,Golden,Musical,5410,85,332180,58,8,False
8056,19,10/19/2003,10,2003,Beauty And The Beast,Lunt-Fontanne,Musical,11285,93,625699,75,8,False
5523,8,4/8/2001,4,2001,Follies,Belasco,Musical,6110,77,332856,67,5,False
8801,20,6/20/2004,6,2004,42nd Street,Ford Center,Musical,11478,79,629795,59,8,False
8686,9,5/9/2004,5,2004,The Boy From Oz,Imperial,Musical,10862,95,812001,85,8,False
4413,2,4/2/2000,4,2000,Cabaret,Studio 54 ('98),Musical,6733,91,463691,83,8,False
3979,14,11/14/1999,11,1999,Fosse,Broadhurst,Musical,8915,98,618434,96,8,False
15363,9,5/9/2010,5,2010,In The Heights,Richard Rodgers,Musical,7888,73,531617,46,8,False
20434,7,12/7/2014,12,2014,Hedwig And The Angry Inch,Belasco,Musical,6002,85,610431,82,7,False


The data wrangling process for this project involved the following transformations:

1. Filtering to keep only rows where Show.Type is "Musical" ensures that we are only analyzing musicals.
2. Standardizing string columns by stripping whitespace from text columns helps in ensuring consistent grouping and reduces errors during analysis.
3. Ensuring numeric columns are correctly typed prevents errors if attendance/revenue were imported as objects due to commas or symbols, allowing for accurate numerical calculations.
4. Removing date suffixes from the 'Show.Name' column ensures all the shows are grouped correctly later during analysis.
5. Fixing title casing of the 'Show.Name' column ensures that all show names are consistently formatted.
6. Identifying and indicating outliers in a separate column allows for easy identification of potential errors or anomalies in the dataset.

Key risks emerging from these transformations include selection bias if certain rows were dropped, or misrepresentation if categorical mappings oversimplified complex attributes. Assumptions were made during cleaning, such as treating specific null values as "unknown" rather than "missing completely at random," which could impact model performance or statistical conclusions. This data is publicly provided by the Broadway League (the national trade association for the Broadway industry), and was aggregated by Austin Cory Bart and distributed via the CORGIS Datasets Project. Ethical acquisition was ensured by confirming that the dataset’s usage license permitted commercial or academic analysis, and the data does not contain PII. To mitigate ethical implications, I would document all transformation steps, conduct sensitivity analyses on how assumptions affect outcomes, and ensure that final outputs do not disproportionately impact any sub-group defined by sensitive attributes like race, gender, or socioeconomic status.